# Model-Profile LogEI Campaign

This notebook demonstrates the v2.1 model-profile workflow using the existing `configs/17_model_profile_logei.yaml` config and `examples/17_model_profile_campaign_log.csv` seed log.

The model profile changes GP fitting behavior only for supported single-objective LogEI/qLogEI campaigns. The CSV schema stays unchanged.

In [ ]:
from pathlib import Path
import math
import shutil

from bo_forge import CampaignSession

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "configs" / "17_model_profile_logei.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "17_model_profile_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "17_model_profile_logei_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "17_model_profile_logei_latest_suggestions.csv"
REPORT_DIR = PROJECT_ROOT / "reports"
TARGET_OBSERVED_ROWS = 15

REPORT_DIR.mkdir(exist_ok=True)
shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)

## Load and validate

The working log is a copy of the committed seed log, so notebook execution never mutates the tracked example data.

In [ ]:
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()

In [ ]:
campaign.model_summary()

## Stage one suggestion

`suggest_next()` is non-mutating. The suggestion is appended only after the explicit `append_suggestions()` call.

In [ ]:
suggestions = campaign.suggest_next(batch_size=1)
suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
suggestions

In [ ]:
campaign.append_suggestions(suggestions)
campaign.pending_suggestions()

## Simulate observations

The objective below is deterministic and local to keep the tutorial reproducible. Replace this function with real experimental measurements in a real campaign.

In [ ]:
def simulate_activity(row):
    loading = float(row["catalyst_loading"])
    temperature = float(row["reaction_temperature"])
    loading_term = 0.62 + 0.85 * loading - 0.92 * (loading - 0.68) ** 2
    temperature_term = -0.00008 * (temperature - 108.0) ** 2
    smooth_variation = 0.035 * math.sin(9.0 * loading + 0.035 * temperature)
    return round(loading_term + temperature_term + smooth_variation, 6)

for row_id in suggestions["row_id"]:
    row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
    campaign.mark_observed(row_id=row_id, objective_value=simulate_activity(row))

campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.observed_data().tail()

## Continue to a compact target

The loop appends and observes one suggested row at a time until the working log reaches `TARGET_OBSERVED_ROWS`.

In [ ]:
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    suggestions = campaign.suggest_next(batch_size=1)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
        campaign.mark_observed(row_id=row_id, objective_value=simulate_activity(row))
    campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)

len(campaign.observed_data())

## Diagnostics and report

`last_fit_*` fields in `model_summary()` are process-local. They show `not_recorded` unless a model fit has happened in the same Python process for matching current fitting inputs.

In [ ]:
campaign.model_summary()

In [ ]:
campaign.plot_model_diagnostics(save_path=REPORT_DIR / "17_model_profile_diagnostics.png")
campaign.export_report(REPORT_DIR / "17_model_profile_report.md")